# initial_ingestion: the one-time history pull (run by hand)

Pulls history from `START_DATE` until now (or `END_DATE`) in 14-day chunks, for every endpoint and area,
and saves the API responses, exactly as received, in the landing volume.

**Run it once, by hand. Never add it to a job.** Edit `START_DATE` before running.

The time in the file names (`..._initial_...`) is when this pull ran, not when EIA published the data,
so these files must not be used to measure how late data arrives.

* If one pull fails, the notebook stops. Fix the problem and run it again: the pipeline ignores rows it
  already has, so repeated data is harmless.
* `landing_ingestion` holds the same helper code. If you change the config or a function, change both.

In [0]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests

In [0]:
# ---------------- CONFIG: edit these ----------------
CATALOG = "dbw_energy_analytics_dev"
SCHEMA = "raw"
VOLUME = "raw_files"

SECRET_SCOPE = "energy-secrets"
API_KEY_SECRET_NAME = "energyg-api-key"     # must match the secret name in Key Vault

REGIONS = ["AVA", "BPAT", "CISO", "PGE", "SCL"]
ENDPOINTS = ["region-data", "fuel-type-data"]
# ----------------------------------------------------

BASE_URL = "https://api.eia.gov/v2/electricity/rto"
HOUR_FMT = "%Y-%m-%dT%H"
MAX_RETRIES = 4
RETRY_WAIT_SECONDS = [5, 15, 30]       # wait before attempt 2, 3 and 4
PAUSE_SECONDS = 1                      # pause between calls

LANDING_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
if not os.path.isdir(LANDING_ROOT):
    raise RuntimeError(f"Volume path not found: {LANDING_ROOT}. Check the names in the config cell.")

In [0]:
def log(level, message, **kwargs):
    print(json.dumps({
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "level": level,
        "message": message,
        **kwargs,
    }))


def get_api_key():
    """Retrieve the EIA API key; fail early with a useful configuration error."""
    try:
        api_key = dbutils.secrets.get(
            scope=SECRET_SCOPE,
            key=API_KEY_SECRET_NAME,
        )

        if not api_key or not api_key.strip():
            raise RuntimeError(
                f"Secret '{API_KEY_SECRET_NAME}' in scope "
                f"'{SECRET_SCOPE}' is empty."
            )

        return api_key

    except RuntimeError:
        raise

    except Exception as e:
        error_type = type(e).__name__

        log(
            "ERROR",
            "EIA API key retrieval failed",
            secret_scope=SECRET_SCOPE,
            secret_name=API_KEY_SECRET_NAME,
            error_type=error_type,
        )

        raise RuntimeError(
            f"Could not retrieve the EIA API key from scope "
            f"'{SECRET_SCOPE}' using secret name "
            f"'{API_KEY_SECRET_NAME}'. Check that the scope and "
            f"secret exist and that you have permission to read it. "
            f"Underlying error: {error_type}."
        ) from e


def fetch(endpoint, region, start, end, api_key):
    """One API call with retries. Returns the response."""
    url = f"{BASE_URL}/{endpoint}/data/"
    params = {
        "api_key": api_key,
        "frequency": "hourly",
        "data[0]": "value",
        "facets[respondent][]": region,
        "start": start,
        "end": end,
        "sort[0][column]": "period",
        "sort[0][direction]": "asc",
        "length": 5000,                  # the most the API returns in one call
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, params=params, timeout=60)
            if response.status_code == 200:
                return response
            log("WARNING", "API returned an error status", endpoint=endpoint, region=region,
                status=response.status_code, attempt=attempt)
            if response.status_code < 500 and response.status_code != 429:
                break                    # bad request or bad key: retrying will not help
        except requests.exceptions.Timeout:
            log("WARNING", "Request timed out", endpoint=endpoint, region=region, attempt=attempt)
        except requests.exceptions.RequestException as e:
            # log only the error type: the full message can contain the URL with the API key
            log("WARNING", "Request failed", endpoint=endpoint, region=region,
                error_type=type(e).__name__, attempt=attempt)
        if attempt < MAX_RETRIES:
            time.sleep(RETRY_WAIT_SECONDS[attempt - 1])
    raise RuntimeError(f"Could not fetch {endpoint} {region} {start} -> {end}")


def pull_and_save(endpoint, region, start, end, window_no, api_key, mode):
    """Fetches one endpoint, area and date range, and saves the response exactly as received."""
    response = fetch(endpoint, region, start, end, api_key)
    data = response.json()
    if "response" not in data:
        raise RuntimeError(f"Unexpected response for {endpoint} {region}")

    total = int(data["response"]["total"])          # the API returns total as text
    rows = len(data["response"]["data"])
    if rows < total:
        # the API only returns 5,000 rows per call: never save an incomplete pull silently
        raise RuntimeError(f"{endpoint} {region}: got {rows} of {total} rows, use a smaller date range")
    if api_key in response.text:
        raise RuntimeError(f"{endpoint} {region}: API key found in the response, not saving it")
    if rows == 0:
        log("WARNING", "API returned zero rows, nothing saved", endpoint=endpoint, region=region,
            start=start, end=end)
        return 0

    now = datetime.now(timezone.utc)
    folder = f"{LANDING_ROOT}/{endpoint}/ingest_date={now.strftime('%Y-%m-%d')}"
    os.makedirs(folder, exist_ok=True)
    path = f"{folder}/{now.strftime('%Y%m%dT%H%M%SZ')}_{mode}_{region}_w{window_no:03d}.json"
    with open(path, "w") as f:
        f.write(response.text)               # raw response, nothing added or removed
    log("INFO", "Saved", endpoint=endpoint, region=region, start=start, end=end, rows=rows, file=path)
    return rows

In [0]:
START_DATE = "2026-10-29"    # first day to pull (UTC), YYYY-MM-DD: edit before running
END_DATE = None              # None = until now, or a day such as "2026-11-05"
CHUNK_DAYS = 14              # one call then returns well under 5,000 rows

now_hour = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0)
start = datetime.strptime(START_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc)
if END_DATE:
    end = datetime.strptime(END_DATE, "%Y-%m-%d").replace(tzinfo=timezone.utc) + timedelta(hours=23)
else:
    end = now_hour
end = min(end, now_hour)
if start > end:
    raise ValueError("START_DATE is after the end of the range")

total_days = (end - start).days + 1
windows = []
for offset in range(0, total_days, CHUNK_DAYS):
    chunk_start = start + timedelta(days=offset)
    chunk_end = min(chunk_start + timedelta(days=CHUNK_DAYS) - timedelta(hours=1), end)
    windows.append((chunk_start.strftime(HOUR_FMT), chunk_end.strftime(HOUR_FMT)))

In [0]:
api_key = get_api_key()
log("INFO", "Run started", mode="initial", windows=len(windows), regions=REGIONS, endpoints=ENDPOINTS)

files_saved, rows_saved = 0, 0
for endpoint in ENDPOINTS:
    for region in REGIONS:
        for window_no, (start, end) in enumerate(windows):
            rows = pull_and_save(endpoint, region, start, end, window_no, api_key, "initial")
            if rows:
                files_saved += 1
                rows_saved += rows
            time.sleep(PAUSE_SECONDS)

log("INFO", "Run finished", mode="initial", files_saved=files_saved, rows_saved=rows_saved)

{"timestamp": "2026-11-07T10:37:50.686105+00:00", "level": "INFO", "message": "Run started", "mode": "initial", "windows": 1, "regions": ["AVA", "BPAT", "CISO", "PGE", "SCL"], "endpoints": ["region-data", "fuel-type-data"]}
{"timestamp": "2026-11-07T10:37:53.169346+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "region": "AVA", "start": "2026-10-29T00", "end": "2026-11-07T10", "rows": 852, "file": "/Volumes/dbw_energy_analytics_dev/raw/raw_files/region-data/ingest_date=2026-11-07/20261107T103752Z_initial_AVA_w000.json"}
{"timestamp": "2026-11-07T10:37:55.728151+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "region": "BPAT", "start": "2026-10-29T00", "end": "2026-11-07T10", "rows": 852, "file": "/Volumes/dbw_energy_analytics_dev/raw/raw_files/region-data/ingest_date=2026-11-07/20261107T103755Z_initial_BPAT_w000.json"}
{"timestamp": "2026-11-07T10:37:58.264373+00:00", "level": "INFO", "message": "Saved", "endpoint": "region-data", "regio